# Tutorial 02: HLLTensor — 2D Tensor View for Disambiguation

This tutorial introduces the `HLLTensor` module, which provides a 2D tensor view of HLLSet fingerprints for disambiguation and semantic operations.

## What You'll Learn

1. **Tensor Structure** — Understanding the (register, zeros) coordinate system
2. **Token Inscription** — Setting bits at specific positions
3. **Active Positions** — Extracting set bit coordinates
4. **Ring Operations** — Union, intersection, XOR on tensors
5. **Token LUT** — Lookup table for disambiguation
6. **BitVector Bridge** — Converting between 1D and 2D views

## Key Concepts

- HLLTensor has shape **(2^p_bits, 32)** — registers × bit positions
- Each position **(reg, zeros)** corresponds to a hash outcome
- Multiple tokens may map to the same position (collisions)
- TokenLUT enables reverse lookup for disambiguation

In [1]:
# Import modules
import sys
sys.path.insert(0, '..')

from core.hll_tensor import HLLTensor, TokenLUT, TokenEntry
from core.hllset import HLLSet, DEFAULT_HASH_CONFIG
from core.bitvector_ring import BitVector
import numpy as np

## 1. Tensor Structure

An HLLTensor is a 2D view of the HLL registers:
- **First dimension**: Register index (0 to 2^p_bits - 1)
- **Second dimension**: Trailing zeros count (0 to 31)

Each register is a 32-bit bitmap where bit `k` is set if an element with `k` trailing zeros was observed.

In [2]:
# Create an empty HLLTensor
tensor = HLLTensor(p_bits=10)

print(f"Precision bits: {tensor.p_bits}")
print(f"Number of registers: {tensor.num_registers}")
print(f"Bits per register: {tensor.bits_per_register}")
print(f"Total bits: {tensor.total_bits}")
print(f"Register shape: {tensor.registers.shape}")
print(f"Register dtype: {tensor.registers.dtype}")

Precision bits: 10
Number of registers: 1024
Bits per register: 32
Total bits: 32768
Register shape: (1024,)
Register dtype: uint32


In [3]:
# Create tensor from existing HLLSet
hll = HLLSet.from_batch(["hello", "world", "foo", "bar"])
tensor_from_hll = HLLTensor.from_numpy(hll.dump_numpy(), p_bits=10)

print(f"Created tensor from HLLSet")
print(f"Popcount: {tensor_from_hll.popcount()}")
print(f"Active registers: {len(tensor_from_hll.active_registers())}")

Created tensor from HLLSet
Popcount: 4
Active registers: 4


## 2. Token Inscription

When a token is added to an HLLSet:
1. Hash the token to get a 64-bit value
2. Extract register index from lower p_bits
3. Count trailing zeros in remaining bits
4. Set bit at position (reg, zeros)

The `inscribe()` method performs step 4.

In [4]:
# Manual inscription example
tensor = HLLTensor(p_bits=10)

# Compute (reg, zeros) for a token
token = "example_token"
reg, zeros = HLLSet.hash_to_reg_zeros(token)
print(f"Token: '{token}'")
print(f"Register: {reg}, Zeros: {zeros}")

# Inscribe the token
tensor.inscribe(reg, zeros)
print(f"\nBit at ({reg}, {zeros}): {tensor.get_bit(reg, zeros)}")
print(f"Popcount after inscription: {tensor.popcount()}")

Token: 'example_token'
Register: 397, Zeros: 2

Bit at (397, 2): True
Popcount after inscription: 1


In [5]:
# Batch inscription
tensor = HLLTensor(p_bits=10)

tokens = ["apple", "banana", "cherry", "date", "elderberry"]
positions = [HLLSet.hash_to_reg_zeros(t) for t in tokens]

tensor.inscribe_batch(positions)

print(f"Inscribed {len(tokens)} tokens")
print(f"Popcount: {tensor.popcount()}")
print(f"Positions: {positions}")

Inscribed 5 tokens
Popcount: 5
Positions: [(385, 1), (852, 2), (354, 0), (912, 3), (410, 2)]


## 3. Active Positions

The `active_positions()` method returns all (reg, zeros) pairs where bits are set.

This is essential for **disambiguation**: each active position represents a potential token that was inscribed.

In [6]:
# Get active positions
tensor = HLLTensor(p_bits=10)

# Inscribe some tokens
for token in ["cat", "dog", "fish", "bird"]:
    reg, zeros = HLLSet.hash_to_reg_zeros(token)
    tensor.inscribe(reg, zeros)

# Retrieve all active positions
active = tensor.active_positions()
print(f"Active positions ({len(active)} total):")
for reg, zeros in active:
    print(f"  Register {reg}, Zeros {zeros}")

Active positions (4 total):
  Register 104, Zeros 1
  Register 269, Zeros 3
  Register 584, Zeros 0
  Register 688, Zeros 1


In [7]:
# Iterator version for memory efficiency
print("Iterating active positions:")
for i, (reg, zeros) in enumerate(tensor.active_positions_iter()):
    print(f"  Position {i}: ({reg}, {zeros})")

Iterating active positions:
  Position 0: (104, 1)
  Position 1: (269, 3)
  Position 2: (584, 0)
  Position 3: (688, 1)


In [8]:
# Active registers (non-zero)
active_regs = tensor.active_registers()
print(f"Active registers: {active_regs}")
print(f"Number of active registers: {len(active_regs)}")

Active registers: [104, 269, 584, 688]
Number of active registers: 4


## 4. Ring Operations

HLLTensor supports bitwise ring operations:

| Operation | Method | Symbol | Description |
|-----------|--------|--------|-------------|
| Union | `union()` | OR | Lattice join |
| Intersection | `intersect()` | AND | Ring multiplication |
| Symmetric Diff | `symmetric_difference()` | XOR | Ring addition |
| Difference | `difference()` | AND-NOT | Set difference |
| Complement | `complement()` | NOT | Bitwise complement |

In [9]:
# Create two tensors
tensor_a = HLLTensor(p_bits=10)
tensor_b = HLLTensor(p_bits=10)

# Inscribe different tokens
for token in ["alpha", "beta", "gamma"]:
    tensor_a.inscribe(*HLLSet.hash_to_reg_zeros(token))

for token in ["gamma", "delta", "epsilon"]:
    tensor_b.inscribe(*HLLSet.hash_to_reg_zeros(token))

print(f"Tensor A popcount: {tensor_a.popcount()}")
print(f"Tensor B popcount: {tensor_b.popcount()}")

Tensor A popcount: 3
Tensor B popcount: 3


In [10]:
# Ring operations
union_ab = tensor_a.union(tensor_b)
intersect_ab = tensor_a.intersect(tensor_b)
xor_ab = tensor_a.symmetric_difference(tensor_b)
diff_ab = tensor_a.difference(tensor_b)

print(f"Union (A ∨ B) popcount: {union_ab.popcount()}")
print(f"Intersection (A ∧ B) popcount: {intersect_ab.popcount()}")
print(f"Symmetric Diff (A ⊕ B) popcount: {xor_ab.popcount()}")
print(f"Difference (A \\ B) popcount: {diff_ab.popcount()}")

Union (A ∨ B) popcount: 5
Intersection (A ∧ B) popcount: 1
Symmetric Diff (A ⊕ B) popcount: 4
Difference (A \ B) popcount: 2


In [11]:
# Subset checking
print(f"A ⊆ Union(A,B): {tensor_a.is_subset(union_ab)}")
print(f"Intersect(A,B) ⊆ A: {intersect_ab.is_subset(tensor_a)}")

A ⊆ Union(A,B): True
Intersect(A,B) ⊆ A: True


## 5. Token Lookup Table (TokenLUT)

The `TokenLUT` maps (reg, zeros) positions back to candidate tokens.

This is the core data structure for **disambiguation**:
- Multiple tokens may hash to the same position (collisions)
- During disambiguation, we retrieve all candidates and triangulate

In [12]:
# Create a TokenLUT
lut = TokenLUT(p_bits=10)

# Add tokens to the LUT
tokens = ["hello", "world", "python", "programming", "data"]

for token in tokens:
    h = HLLSet.hash(token)
    reg, zeros = HLLSet.hash_to_reg_zeros(token)
    lut.add_token(token, reg, zeros, hash_full=h, layer=0)

print(f"Added {len(tokens)} tokens to LUT")
print(f"Unique positions: {len(lut.positions())}")

Added 5 tokens to LUT
Unique positions: 5


In [13]:
# Lookup candidates at a position
test_token = "hello"
reg, zeros = HLLSet.hash_to_reg_zeros(test_token)

candidates = lut.lookup(reg, zeros)
print(f"Position ({reg}, {zeros}) candidates:")
for entry in candidates:
    print(f"  Token: '{entry.token}', Layer: {entry.layer}")

Position (135, 1) candidates:
  Token: 'hello', Layer: 0


In [14]:
# Add n-grams (multi-word tokens) with layer information
ngrams = [
    ("machine learning", 1),      # bigram
    ("deep learning", 1),         # bigram  
    ("natural language processing", 2),  # trigram
]

for ngram, layer in ngrams:
    h = HLLSet.hash(ngram)
    reg, zeros = HLLSet.hash_to_reg_zeros(ngram)
    lut.add_token(ngram, reg, zeros, hash_full=h, layer=layer)

# Query by layer
print("Unigrams (layer 0):")
for entry in lut.entries_at_layer(0):
    print(f"  {entry.token}")

print("\nBigrams (layer 1):")
for entry in lut.entries_at_layer(1):
    print(f"  {entry.token}")

Unigrams (layer 0):
  hello
  world
  python
  programming
  data

Bigrams (layer 1):
  machine learning
  deep learning


In [15]:
# Generate n-grams from long text using sliding window
def sliding_window_ngrams(text: str, window_size: int = 3) -> list:
    """
    Generate n-grams from text using a sliding window.
    
    Args:
        text: Input text (will be split on whitespace)
        window_size: Number of tokens per n-gram (default: 3)
    
    Returns:
        List of (ngram_string, layer) tuples
    """
    words = text.split()
    ngrams = []
    
    # Generate n-grams for each layer (1 to window_size)
    for n in range(1, window_size + 1):
        layer = n - 1  # layer 0 = unigrams, layer 1 = bigrams, etc.
        for i in range(len(words) - n + 1):
            ngram = " ".join(words[i:i + n])
            ngrams.append((ngram, layer))
    
    return ngrams

# Example: Process a paragraph
sample_text = """
The quick brown fox jumps over the lazy dog. 
Machine learning models process natural language effectively.
"""

# Clean and normalize
clean_text = " ".join(sample_text.lower().split())
print(f"Input text: {clean_text[:60]}...")

# Generate n-grams with window size 3 (unigrams, bigrams, trigrams)
all_ngrams = sliding_window_ngrams(clean_text, window_size=3)

print(f"\nGenerated {len(all_ngrams)} n-grams:")
print(f"  Unigrams (layer 0): {len([ng for ng in all_ngrams if ng[1] == 0])}")
print(f"  Bigrams (layer 1): {len([ng for ng in all_ngrams if ng[1] == 1])}")
print(f"  Trigrams (layer 2): {len([ng for ng in all_ngrams if ng[1] == 2])}")

# Add all to the LUT
lut_text = TokenLUT(p_bits=10)
for ngram, layer in all_ngrams:
    h = HLLSet.hash(ngram)
    reg, zeros = HLLSet.hash_to_reg_zeros(ngram)
    lut_text.add_token(ngram, reg, zeros, hash_full=h, layer=layer)

print(f"\nLUT now has {len(lut_text.positions())} unique positions")

# Show some examples from each layer
print("\nSample n-grams:")
for layer in range(3):
    entries = lut_text.entries_at_layer(layer)[:3]  # First 3
    layer_name = ["Unigrams", "Bigrams", "Trigrams"][layer]
    print(f"  {layer_name}: {[e.token for e in entries]}")

Input text: the quick brown fox jumps over the lazy dog. machine learnin...

Generated 45 n-grams:
  Unigrams (layer 0): 16
  Bigrams (layer 1): 15
  Trigrams (layer 2): 14

LUT now has 44 unique positions

Sample n-grams:
  Unigrams: ['the', 'the', 'quick']
  Bigrams: ['the quick', 'quick brown', 'brown fox']
  Trigrams: ['the quick brown', 'quick brown fox', 'brown fox jumps']


## 6. BitVector Bridge

HLLTensor can convert to/from `BitVector` for ring layer operations.

- **BitVector**: 1D view for pure bitwise ring algebra
- **HLLTensor**: 2D view for semantic disambiguation

In [16]:
# Create tensor and convert to BitVector
tensor = HLLTensor(p_bits=10)
for token in ["bit", "vector", "test"]:
    tensor.inscribe(*HLLSet.hash_to_reg_zeros(token))

# Convert to 1D BitVector
bv = tensor.to_bitvector()
print(f"BitVector N: {bv.N}")
print(f"BitVector popcount: {bv.popcount()}")
print(f"BitVector value (hex, first 32 bits): {hex(bv.value & 0xFFFFFFFF)}")

BitVector N: 32768
BitVector popcount: 3
BitVector value (hex, first 32 bits): 0x0


In [17]:
# Convert BitVector back to tensor
tensor_restored = HLLTensor.from_bitvector(bv, p_bits=10)

print(f"Original popcount: {tensor.popcount()}")
print(f"Restored popcount: {tensor_restored.popcount()}")
print(f"Tensors equal: {tensor == tensor_restored}")

Original popcount: 3
Restored popcount: 3
Tensors equal: True


## 7. Cardinality Support

HLLTensor provides methods to support HLL cardinality estimation.

In [18]:
# Create tensor with many tokens
tensor = HLLTensor(p_bits=10)
for i in range(500):
    tensor.inscribe(*HLLSet.hash_to_reg_zeros(f"token_{i}"))

# Get max zeros per register (for HLL estimation)
max_zeros = tensor.max_zeros_per_register()
print(f"Max zeros array shape: {max_zeros.shape}")
print(f"Max zeros statistics:")
print(f"  Mean: {max_zeros.mean():.2f}")
print(f"  Max: {max_zeros.max()}")
print(f"  Non-zero count: {np.count_nonzero(max_zeros)}")

Max zeros array shape: (1024,)
Max zeros statistics:
  Mean: 0.86
  Max: 10
  Non-zero count: 400


In [19]:
# Per-register popcount
active_regs = tensor.active_registers()[:5]  # First 5 active
print("Per-register popcount (first 5 active):")
for reg in active_regs:
    pc = tensor.register_popcount(reg)
    print(f"  Register {reg}: {pc} bits set")

Per-register popcount (first 5 active):
  Register 4: 1 bits set
  Register 5: 1 bits set
  Register 9: 1 bits set
  Register 10: 1 bits set
  Register 11: 1 bits set


### Cardinality Inflation with N-grams

**Important**: When using sliding window n-grams, the HLLSet cardinality is **inflated** by approximately a factor of `n` (the window size).

For a text with `W` unique words and window size `n`:
- Unigrams: ~W entries
- Bigrams: ~W entries  
- Trigrams: ~W entries
- **Total**: ~n × W entries

The **lower bound** estimate for true unique words:
$$\text{Unique Words} \approx \frac{\text{cardinality}(\text{HLLSet})}{n}$$

This is useful for normalizing cardinality when comparing HLLSets built with different n-gram strategies.

In [20]:
# Demonstrate cardinality inflation with n-grams
# Reuse the sliding_window_ngrams function and sample text from earlier

# Count actual unique words in the text
unique_words = set(clean_text.split())
actual_unique = len(unique_words)

# Build HLLSet with all n-grams (window_size=3)
window_size = 3
hll_ngrams = HLLSet.from_batch([ng[0] for ng in all_ngrams])
inflated_cardinality = hll_ngrams.cardinality()

# Calculate lower bound estimate
lower_bound = inflated_cardinality / window_size

print(f"Text statistics:")
print(f"  Actual unique words: {actual_unique}")
print(f"  Window size (n): {window_size}")
print(f"")
print(f"HLLSet cardinality:")
print(f"  Inflated (all n-grams): ~{inflated_cardinality:.0f}")
print(f"  Lower bound estimate:   ~{lower_bound:.0f}")
print(f"  Actual unique words:     {actual_unique}")
print(f"")
print(f"Inflation factor: {inflated_cardinality / actual_unique:.2f}x")
print(f"(Expected ~{window_size}x due to {window_size}-gram window)")

Text statistics:
  Actual unique words: 15
  Window size (n): 3

HLLSet cardinality:
  Inflated (all n-grams): ~49
  Lower bound estimate:   ~16
  Actual unique words:     15

Inflation factor: 3.27x
(Expected ~3x due to 3-gram window)


## 8. Visualization

HLLTensor can display a region as ASCII art for debugging.

In [21]:
# Visualize a region of the tensor
tensor = HLLTensor(p_bits=10)

# Inscribe some tokens
for token in ["vis", "test", "demo", "show", "display"]:
    tensor.inscribe(*HLLSet.hash_to_reg_zeros(token))

# Show first 15 registers and 8 zero positions
tensor.show_region(max_reg=15, max_zeros=8)

HLLTensor[:15, :8] (popcount=5)
      0  1  2  3  4  5  6  7
     ────────────────────────
  0 │ ·  ·  ·  ·  ·  ·  ·  · 
  1 │ ·  ·  ·  ·  ·  ·  ·  · 
  2 │ ·  ·  ·  ·  ·  ·  ·  · 
  3 │ ·  ·  ·  ·  ·  ·  ·  · 
  4 │ ·  ·  ·  ·  ·  ·  ·  · 
  5 │ ·  ·  ·  ·  ·  ·  ·  · 
  6 │ ·  ·  ·  ·  ·  ·  ·  · 
  7 │ ·  ·  ·  ·  ·  ·  ·  · 
  8 │ ·  ·  ·  ·  ·  ·  ·  · 
  9 │ ·  ·  ·  ·  ·  ·  ·  · 
 10 │ ·  ·  ·  ·  ·  ·  ·  · 
 11 │ ·  ·  ·  ·  ·  ·  ·  · 
 12 │ ·  ·  ·  ·  ·  ·  ·  · 
 13 │ ·  ·  ·  ·  ·  ·  ·  · 
 14 │ ·  ·  ·  ·  ·  ·  ·  · 


## Summary

In this tutorial, you learned:

1. **Tensor Structure** — 2D array of (2^p_bits × 32) for HLL registers
2. **Token Inscription** — Setting bits at (reg, zeros) positions
3. **Active Positions** — Extracting coordinates for disambiguation
4. **Ring Operations** — Union, intersection, XOR, difference
5. **TokenLUT** — Reverse lookup for candidate tokens
6. **BitVector Bridge** — Converting between 1D and 2D views

### Next Steps

- **Tutorial 03**: HLLLattice — Temporal lattice structures
- **Tutorial 04**: HLLSetStore — Persistent storage with derivation tracking